# Unsloth Studio (X8) on a free Colab T4

Clones **https://github.com/GoldodemonINC/unsloth_8X.git** (branch `x8-main`),
installs Unsloth Studio (backend + built frontend), starts it on port 8888 and
shows the UI through **Colab's own port proxy**.

**Runtime -> Change runtime type -> T4 GPU** (the free tier is enough).

1. Check the GPU
2. Clone the repo (public - no credentials needed)
3. Install (this builds the frontend - expect a few minutes)
4. Start Studio - one cell, which always prints a shareable `https://` link


In [ ]:
import sys, torch

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device :", props.name, "- %.1f GB" % (props.total_memory / 1024 ** 3))


## 2. Clone the repo

The repository is **public**, so the clone works as-is.

Optionally add a `GITHUB_TOKEN` secret in Colab (the key icon, *Secrets*) if
you hit GitHub's anonymous rate limit - the cell below picks it up when it
exists and ignores it otherwise.

In [ ]:
REPO_URL  = "https://github.com/GoldodemonINC/unsloth_8X.git"
# The branch this working tree is pushed to. If it is missing on the remote the
# clone falls back to main, so the cell still runs on a fresh fork.
BRANCH   = "x8-main"
REPO_DIR = "/content/unsloth_8X"

import shutil, subprocess

# The repo is public, so no credentials are needed. A GITHUB_TOKEN secret is
# used when present (it lifts GitHub's anonymous rate limit).
CLONE_URL = REPO_URL
token = None
try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if token:
    CLONE_URL = REPO_URL.replace("https://", "https://%s@" % token)
    print("using the GITHUB_TOKEN secret from Colab")
else:
    print("no GITHUB_TOKEN secret - cloning the public repo anonymously")


def clone(branch):
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    return subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", branch, CLONE_URL, REPO_DIR],
        capture_output=True, text=True,
    )


used_branch = BRANCH
result = clone(used_branch)
if result.returncode != 0 and BRANCH != "main":
    print("branch %s is not on the remote; falling back to main" % BRANCH)
    used_branch = "main"
    result = clone(used_branch)

for stream in (result.stdout, result.stderr):
    text = stream or ""
    if token:
        text = text.replace(token, "***")
    print(text[-2000:])

if result.returncode != 0:
    raise SystemExit(
        "Clone failed. The repo is public, so this is usually a network hiccup "
        "- re-run this cell. If it keeps failing, add a GITHUB_TOKEN secret "
        "(Colab -> Secrets) and run it again."
    )

sha = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()
print("checked out %s @ %s" % (used_branch, sha[:12]))
print("Compare this with your local `git rev-parse HEAD` before trusting a run.")

## 3. Install Unsloth Studio

Backend dependencies go into Colab's system Python (no venv on Colab) and the
frontend is built into `studio/frontend/dist`, which is what the server serves.


In [ ]:
%%bash
cd /content/unsloth_8X
chmod +x studio/setup.sh
# whisper.cpp / audio.cpp are not needed for training or merging and only slow
# the install down on Colab, so they are skipped.
UNSLOTH_SKIP_WHISPER_INSTALL=1 UNSLOTH_SKIP_AUDIO_CPP_INSTALL=1 bash studio/setup.sh --local


## 4. Start Studio (one cell, shareable link)

This single cell starts the backend on port 8888 and always opens a **shareable
`https://` link** (a Cloudflare quick tunnel) that works from a phone or any
browser tab. Colab's own port-proxy URLs are session-scoped and usually stay
blank in the cell, so they are informational only.

Log in as **`unsloth`** with the password printed in the link card. Leave the
cell running - the server runs in its own thread, and stops when you interrupt
it (**Kernel -> Interrupt**).

In [ ]:
import sys

REPO_DIR = "/content/unsloth_8X"
BACKEND  = f"{REPO_DIR}/studio/backend"
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

import colab

PORT = 8888

# One cell, one call: start the server and open the shareable tunnel. cloudflare=True is
# forced rather than left to detection, so the link is always shareable even if the runtime
# check cannot run. The admin password is settled first and printed in the link card.
colab.start(port=PORT, cloudflare=True)


## 5. Smoke test the Hub path used by Merging Studio

This is the same code the Merging page calls: it reads only `config.json` from
the Hub, so it is quick and downloads no weights.

In [ ]:
import sys
sys.path.insert(0, "/content/unsloth_8X/studio/backend")
from core.advanced import merge

repo = "unsloth/Llama-3.2-1B"
print("repo id :", merge.hf_repo_id(repo))
print("spec    :", merge.probe_model_spec(f"hf://{repo}"))


###Troubleshooting
* **"Frontend not built"** - re-run the install cell (step 3).
* **"Clone failed"** - the repo is public, so re-run the cell; if it keeps
  failing, add a `GITHUB_TOKEN` secret in Colab (*Secrets* panel).
* **No shareable link** - the card prints why (usually the tunnel binary failing to
  download). Re-run the start cell once the logs above settle; the server itself is
  unaffected and keeps running.
* **Port already in use** - set `PORT = 8889` in the start cell before calling `start()`.
* **Restarting** - *Runtime -> Restart runtime*, then run the cells again.
* **Testing the merge search** - in Studio open **Advanced -> Merging**, leave each
  slot in **Hugging Face** mode and type e.g. `llama 3.2 1b`. The list comes from
  the Hub, the same search the trainer uses.
